# ЛР3. Клиент

Тетрадь подготовлена на основе предыдущей ЛР и требований ЛР3.

In [ ]:
import xmlrpc.client
import pickle
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
%matplotlib inline

# Вспомогательная функция для отображения результатов
def display_results(original, processed, operation_title):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.title("Оригинал")
    cmap_type = 'gray' if len(original.shape) == 2 or original.shape[-1] == 1 else None
    plt.imshow(original, cmap=cmap_type)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title(operation_title)
    cmap_type_proc = 'gray' if len(processed.shape) == 2 or processed.shape[-1] == 1 else None
    plt.imshow(processed, cmap=cmap_type_proc)
    plt.axis('off')
    plt.show()

# Подключение к серверам
server = xmlrpc.client.ServerProxy("http://localhost:8010", allow_none=True)
stats = xmlrpc.client.ServerProxy("http://localhost:8009", allow_none=True)

print("="*60)
print("ТЕСТ 1: Базовые функции")
print("="*60)
print("Методы прокси:", server.system.listMethods())
print("\nPing:", server.ping())
print("Now:", server.now())
print("Type:", server.show_type(2))
print("Sum:", server.test_sum(2, 3))
print("Pow:", server.test_pow(2, 3))

print("\n" + "="*60)
print("ТЕСТ 2: Проверка чёрного списка")
print("="*60)
test_cases = [
    ('Иванов', 'Иван', 'Иванович', '22.03.1989'),
    ('иванов', 'иван', 'иванович', '22.03.1989'),  # регистр
    ('Иваноа', 'Иван', 'Иванович', '22.03.1989'),  # опечатка
    ('Сидоров', 'Пётр', 'Петрович', '01.01.1990')  # не в списке
]
for args in test_cases:
    result = server.black_list_check(*args)
    print(f"{args} => {result}")

print("\n" + "="*60)
print("ТЕСТ 3: Обработка изображений")
print("="*60)

# Загрузка изображений
img_color = np.array(Image.open('Jellyfish.jpg'))
img_mono = np.array(Image.open('11.bmp'))

# Упаковка в бинарный формат
pimg_color = xmlrpc.client.Binary(pickle.dumps(img_color))
pimg_mono = xmlrpc.client.Binary(pickle.dumps(img_mono))
threshold = 40

# 3.1 Инверсия цвета
print("\n3.1. Инверсия цвета")
inv_color = pickle.loads(server.color_inversion(pimg_color).data)
inv_mono = pickle.loads(server.color_inversion(pimg_mono).data)
display_results(img_color, inv_color, "Инверсия (Цветное)")
display_results(img_mono, inv_mono, "Инверсия (Монохромное)")

# 3.2 Бинаризация
print("\n3.2. Бинаризация изображения")
bw_color = pickle.loads(server.binarize_image(pimg_color, threshold).data)
bw_mono = pickle.loads(server.binarize_image(pimg_mono, threshold).data)
display_results(img_color, bw_color, f"Бинаризация (Порог {threshold})")
display_results(img_mono, bw_mono, f"Бинаризация (Порог {threshold})")

# 3.3 Разворот по вертикали
print("\n3.3. Разворот по вертикали")
flip_color = pickle.loads(server.flip_vertical(pimg_color).data)
flip_mono = pickle.loads(server.flip_vertical(pimg_mono).data)
display_results(img_color, flip_color, "Разворот (Цветное)")
display_results(img_mono, flip_mono, "Разворот (Монохромное)")

# 3.4 Процент облачности
print("\n3.4. Оценка процента облачности")
print(f"Облачность color: {server.cloudiness_percent(pimg_color, threshold):.2f}%")
print(f"Облачность mono: {server.cloudiness_percent(pimg_mono, threshold):.2f}%")

print("\n" + "="*60)
print("ТЕСТ 4: Rate Limiting (5 запросов в 5 сек)")
print("="*60)
print("Выполняем 6 вызовов ping подряд:")
for i in range(6):
    result = server.ping()
    print(f"  Вызов {i+1}: {result}")

print("\n" + "="*60)
print("ТЕСТ 5: Журнал событий и статистика")
print("="*60)

print("\n5.1. Последние 20 событий:")
events = stats.get_events('', '', '', '', '')
for e in events[-20:]:
    print(f"  {e}")

print("\n5.2. Срез по типу события 'test_sum':")
sum_events = stats.get_events('test_sum', '', '', '', '')
print(f"  Найдено событий: {len(sum_events)}")
for e in sum_events[:5]:
    print(f"  {e}")

print("\n5.3. Срез по времени (2020-2030):")
time_events = stats.get_events('', '2020-01-01 00:00:00', '2030-12-31 23:59:59.999999', '', '')
print(f"  Найдено событий: {len(time_events)}")
for e in time_events[:5]:
    print(f"  {e}")

print("\n5.4. Срез по длительности (duration >= 0):")
duration_events = stats.get_events('', '', '', '0', '')
print(f"  Найдено событий: {len(duration_events)}")
for e in duration_events[:5]:
    print(f"  {e}")

print("\n5.5. Общая статистика:")
stats_data = stats.get_stats()
print(f"  Всего событий: {stats_data['count']}")
print(f"  Средняя длительность: {stats_data['avg_duration']:.6f} сек")
print(f"  Минимальная длительность: {stats_data['min_duration']:.6f} сек")
print(f"  Максимальная длительность: {stats_data['max_duration']:.6f} сек")

print("\n" + "="*60)
print("ТЕСТ 6: Отказоустойчивость")
print("="*60)
print("Остановите stats-сервер и повторите server.test_sum(2, 3).")
print("Результат должен остаться 5, так как прокси не делает статистику обязательной.")
print("\nВсе тесты выполнены!")
